# 02i — Support User Feedback (by type)

Procesa `data/data_raw/support_user_feedback_by_type.csv` para generar
features de feedback explícito por usuario.

**CSV de entrada:** 2.18 MB, 16.664 filas, 8 columnas:
`_id`, `user_id`, `game_version`, `platform`, `feedback_type`, `state`,
`updated_at`, `created_at`.

**Cobertura esperada:** ~652 usuarios del sample (0.52%). Es un CSV
pequeño en volumen pero **cualitativamente único**: contiene señales
explícitas del usuario sobre su experiencia (bugs, dificultad, queja
sobre precios, satisfacción).

**Hallazgos clave:**
- 27 valores únicos de `feedback_type` agrupados manualmente en 5
  categorías: `positive`, `negative`, `bug`, `monetization`, `neutral`.
- La categoría `monetization` (`MEMBERSHIP_EXPENSIVE`, `CANT_PAY`) es de
  bajo volumen pero alta señal predictiva — usuarios que se quejan del
  precio churnean fuerte.
- 0% user_id NaN (limpio).

**Política point-in-time:** descartar filas con `created_at > REFERENCE_DATE`
o `updated_at > REFERENCE_DATE`.

**Outputs:**
- `data/data_qc/features_feedback.parquet` (126.253 × 7)
- `informes/fase1_cleaning/support_feedback/execution_report.md`
- HTML + sección enriquecida

In [1]:
# [SETUP] Imports y dependencias
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gc
import time
from pathlib import Path
from datetime import datetime

plt.style.use('default')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

In [2]:
# [SETUP] Paths, constantes y helpers
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase1_cleaning' else Path.cwd()
DATA_RAW = PROJECT_ROOT / 'data' / 'data_raw'
DATA_QC = PROJECT_ROOT / 'data' / 'data_qc'
REPORT_DIR = PROJECT_ROOT / 'informes' / 'fase1_cleaning' / 'support_feedback'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

CSV_INPUT = DATA_RAW / 'support_user_feedback_by_type.csv'
SAMPLE_IDS_PATH = DATA_QC / 'sample_user_ids_cutoff.parquet'
PARQUET_FEATURES = DATA_QC / 'features_feedback_cutoff.parquet'

def clean_user_id(uid):
    uid = str(uid)
    if uid.startswith('ObjectId(') and uid.endswith(')'):
        return uid[9:-1].replace("'", "")
    return uid

steps_log = []
NOTEBOOK_START = time.time()

def log_step(tag, message):
    ts = datetime.now().strftime('%H:%M:%S')
    entry = f"[{tag}] {ts} — {message}"
    steps_log.append(entry)
    print(entry)

# Mapeo manual de feedback_types → categoría (5 valores)
FEEDBACK_CATEGORY_MAP = {
    # Positivos (34.4% del volumen total)
    'VERY_FUN': 'positive',
    'LOVE_ITEMS': 'positive',
    'ENEMIES_FLY_LOVE': 'positive',
    'GREAT_VISUALS': 'positive',
    'LIKE': 'positive',
    'FEM_CHARACTERS': 'positive',
    'ALL_GAME_ACCESIBLE': 'positive',

    # Negativos de gameplay (29.8%)
    'OBTAIN_ITEMS_EASIER': 'negative',
    'DONT_LIKE': 'negative',
    'TOO_MANY_HITS': 'negative',
    'DIFFICULT_GAME': 'negative',
    'ENEMIES_FLY_HATE': 'negative',
    'DODGE_DIFFICULT': 'negative',
    'NOT_RESPONSIVE_CONTROL': 'negative',
    'LITTLE_ENERGY_MANA': 'negative',
    'MEDALS_BALANCE': 'negative',

    # Bugs / errores técnicos (19.7%)
    'OTHER_BUGS': 'bug',
    'GEMS_ERROR': 'bug',
    'ENERGY_MANA_NOT_FOUND': 'bug',
    'WEEKLY_CHEST_ERROR': 'bug',
    'STATS_ERROR': 'bug',
    'VIDEO_NOT_REWARDED': 'bug',
    'VERIFY_EMAIL_ERROR': 'bug',
    'RECOVER_ACCOUNT_ERROR': 'bug',

    # Monetización (3.5%, joya predictiva)
    'MEMBERSHIP_EXPENSIVE': 'monetization',
    'CANT_PAY': 'monetization',

    # Neutral
    'SUGGESTION': 'neutral',
}

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"CSV_INPUT existe: {CSV_INPUT.exists()}")
print(f"FEEDBACK_CATEGORY_MAP: {len(FEEDBACK_CATEGORY_MAP)} feedback_types mapeados")

PROJECT_ROOT: /Users/jezquerro/Documents/tfg
CSV_INPUT existe: True
FEEDBACK_CATEGORY_MAP: 27 feedback_types mapeados


In [3]:
# [SETUP] Carga sample_user_ids, REFERENCE_DATE y sentinel
sample_ids_df = pd.read_parquet(SAMPLE_IDS_PATH)
sample_ids_df['user_id'] = sample_ids_df['user_id'].apply(clean_user_id)
sample_user_ids = set(sample_ids_df['user_id'])
N_SAMPLE = len(sample_user_ids)

sample_example = list(sample_user_ids)[0]
assert len(sample_example) == 24 and not sample_example.startswith('ObjectId'), \
    f"ERROR: user_id no es hex limpio: '{sample_example}'"

users_clean = pd.read_parquet(DATA_QC / 'users_clean_cutoff.parquet', columns=['last_login_dt'])
REFERENCE_DATE = users_clean['last_login_dt'].max()
CUTOFF_DATE = REFERENCE_DATE - pd.Timedelta(days=120)
del users_clean
gc.collect()

REFERENCE_DATE_UNIX = int(REFERENCE_DATE.timestamp())
CUTOFF_DATE_UNIX = int(CUTOFF_DATE.timestamp())
SENTINEL_DAYS = 9999

print(f"Usuarios en sample: {N_SAMPLE:,}")
print(f"REFERENCE_DATE: {REFERENCE_DATE}")
print(f"REFERENCE_DATE_UNIX: {REFERENCE_DATE_UNIX}")
print(f"SENTINEL_DAYS: {SENTINEL_DAYS}")

log_step('SETUP', f'sample_user_ids: {N_SAMPLE:,} usuarios')
log_step('SETUP', f'REFERENCE_DATE_UNIX: {REFERENCE_DATE_UNIX}')

Usuarios en sample: 25,200
REFERENCE_DATE: 2026-04-04 18:23:13
REFERENCE_DATE_UNIX: 1775326993
SENTINEL_DAYS: 9999
[SETUP] 13:13:35 — sample_user_ids: 25,200 usuarios
[SETUP] 13:13:35 — REFERENCE_DATE_UNIX: 1775326993


## 1. Carga y exploración del CSV

In [4]:
# [EXEC] Cargar support_user_feedback_by_type.csv
t0 = time.time()
df_raw = pd.read_csv(CSV_INPUT, low_memory=False)
load_time = time.time() - t0

print(f"Shape: {df_raw.shape}")
print(f"Tiempo: {load_time:.1f}s")
print(f"Memoria: {df_raw.memory_usage(deep=True).sum() / 1e6:.2f} MB")
log_step('EXEC', f'Carga CSV: shape={df_raw.shape}, tiempo={load_time:.1f}s')

Shape: (16664, 8)
Tiempo: 0.0s
Memoria: 3.17 MB
[EXEC] 13:13:35 — Carga CSV: shape=(16664, 8), tiempo=0.0s


In [5]:
# [ANALYSIS] Tipos de datos del CSV crudo
print("TIPOS DE DATOS — feedback raw")
print("=" * 80)
for col in df_raw.columns:
    dtype = df_raw[col].dtype
    n_nulls = df_raw[col].isnull().sum()
    pct_nulls = n_nulls / len(df_raw) * 100
    n_unique = df_raw[col].nunique()
    example = df_raw[col].dropna().iloc[0] if n_nulls < len(df_raw) else 'TODO NULO'
    print(f"  {col:<20} dtype={str(dtype):<10} nulls={n_nulls:>8,} ({pct_nulls:5.1f}%)  "
          f"unique={n_unique:>8,}  ej='{str(example)[:30]}'")

# Nulos por columna → CSV
nulls_raw = df_raw.isnull().sum().to_frame(name='n_nulls')
nulls_raw['pct_nulls'] = (nulls_raw['n_nulls'] / len(df_raw) * 100).round(2)
nulls_raw = nulls_raw.sort_values('pct_nulls', ascending=False)
nulls_raw.to_csv(REPORT_DIR / 'nulos_por_columna_raw.csv')

# % user_id NaN sobre CSV COMPLETO (esperado 0%)
pct_uid_nan = df_raw['user_id'].isna().mean() * 100
print(f"\n% user_id NaN (CSV completo): {pct_uid_nan:.3f}%")
log_step('ANALYSIS', f'Feedback raw: {pct_uid_nan:.3f}% NaN, '
         f'{df_raw["feedback_type"].nunique()} feedback_types únicos')

TIPOS DE DATOS — feedback raw
  _id                  dtype=str        nulls=       0 (  0.0%)  unique=  16,664  ej='ObjectId(635a9246d45f3b4dd70c3'
  user_id              dtype=str        nulls=       0 (  0.0%)  unique=   8,750  ej='6193ea8fd66f0358fe2616ea'
  game_version         dtype=int64      nulls=       0 (  0.0%)  unique=      48  ej='17'
  platform             dtype=str        nulls=       0 (  0.0%)  unique=       2  ej='android'
  feedback_type        dtype=str        nulls=       0 (  0.0%)  unique=      27  ej='TOO_MANY_HITS'
  state                dtype=int64      nulls=       0 (  0.0%)  unique=       2  ej='0'
  updated_at           dtype=str        nulls=       0 (  0.0%)  unique=  16,664  ej='2022-10-27T14:14:30.233Z'
  created_at           dtype=str        nulls=       0 (  0.0%)  unique=  16,664  ej='2022-10-27T14:14:30.233Z'

% user_id NaN (CSV completo): 0.000%
[ANALYSIS] 13:13:35 — Feedback raw: 0.000% NaN, 27 feedback_types únicos


In [6]:
# [ANALYSIS] Distribuciones clave del CSV crudo
print("=== feedback_type (todos los valores) ===")
fb_dist = df_raw['feedback_type'].value_counts(dropna=False)
for ft, n in fb_dist.items():
    pct = n / len(df_raw) * 100
    print(f"  {ft:<28} {n:>6,} ({pct:5.2f}%)")
print(f"\nTotal feedback_types únicos: {df_raw['feedback_type'].nunique()}")

# Guardar distribución
fb_dist.to_csv(REPORT_DIR / 'distribucion_feedback_type_raw.csv')

print("\n=== state ===")
print(df_raw['state'].value_counts(dropna=False))

print("\n=== platform ===")
print(df_raw['platform'].value_counts(dropna=False))

# Distribución feedbacks/usuario
fb_per_user = df_raw.groupby('user_id').size()
print(f"\n=== Feedbacks por usuario ===")
print(f"  Mediana: {fb_per_user.median():.0f}")
print(f"  Media:   {fb_per_user.mean():.2f}")
print(f"  Min:     {fb_per_user.min()}")
print(f"  Max:     {fb_per_user.max()}")
print(f"  P90:     {fb_per_user.quantile(0.9):.0f}")
print(f"  P99:     {fb_per_user.quantile(0.99):.0f}")
print(f"\nTop 10 usuarios por feedback count:")
print(fb_per_user.sort_values(ascending=False).head(10))

log_step('ANALYSIS', f'Feedbacks/user mediana={fb_per_user.median():.0f}, max={fb_per_user.max()}')

=== feedback_type (todos los valores) ===
  SUGGESTION                    2,106 (12.64%)
  VERY_FUN                      2,046 (12.28%)
  OBTAIN_ITEMS_EASIER           1,348 ( 8.09%)
  OTHER_BUGS                      916 ( 5.50%)
  LOVE_ITEMS                      855 ( 5.13%)
  ENEMIES_FLY_LOVE                835 ( 5.01%)
  DONT_LIKE                       748 ( 4.49%)
  GREAT_VISUALS                   712 ( 4.27%)
  GEMS_ERROR                      636 ( 3.82%)
  ALL_GAME_ACCESIBLE              588 ( 3.53%)
  NOT_RESPONSIVE_CONTROL          574 ( 3.44%)
  LITTLE_ENERGY_MANA              531 ( 3.19%)
  DODGE_DIFFICULT                 530 ( 3.18%)
  MEMBERSHIP_EXPENSIVE            512 ( 3.07%)
  ENERGY_MANA_NOT_FOUND           504 ( 3.02%)
  FEM_CHARACTERS                  452 ( 2.71%)
  TOO_MANY_HITS                   417 ( 2.50%)
  VIDEO_NOT_REWARDED              417 ( 2.50%)
  DIFFICULT_GAME                  348 ( 2.09%)
  MEDALS_BALANCE                  315 ( 1.89%)
  WEEKLY_CHEST_ERR

## 2. Categorización + filtrado point-in-time

Tres pasos:
1. Mapear `feedback_type` a 5 categorías (`positive`, `negative`, `bug`,
   `monetization`, `neutral`).
2. Parsear timestamps con epoch + Timedelta (independiente de la unidad).
3. Aplicar filtros point-in-time + sample_user_ids.

In [7]:
# [EXEC] Categorización: feedback_type → feedback_category
# CRÍTICO: assert de cobertura completa del mapeo (detecta tipos nuevos).
actual_types = set(df_raw['feedback_type'].dropna().unique())
mapped_types = set(FEEDBACK_CATEGORY_MAP.keys())
new_types = actual_types - mapped_types
assert not new_types, (
    f"Hay feedback_types sin categorizar: {new_types}. "
    "Actualizar FEEDBACK_CATEGORY_MAP en este notebook."
)
print(f"  Cobertura del mapeo: {len(actual_types)}/{len(actual_types)} tipos cubiertos")

df_raw['feedback_category'] = df_raw['feedback_type'].map(FEEDBACK_CATEGORY_MAP)

# Distribución por categoría
print("\n=== Distribución por categoría ===")
cat_dist = df_raw['feedback_category'].value_counts(dropna=False)
for cat, n in cat_dist.items():
    pct = n / len(df_raw) * 100
    print(f"  {cat:<15} {n:>6,} ({pct:5.2f}%)")

cat_dist.to_csv(REPORT_DIR / 'distribucion_categoria_raw.csv')
log_step('EXEC', f'Categorización aplicada: {len(actual_types)} tipos → {len(set(FEEDBACK_CATEGORY_MAP.values()))} categorías')

  Cobertura del mapeo: 27/27 tipos cubiertos

=== Distribución por categoría ===
  positive         5,736 (34.42%)
  negative         4,964 (29.79%)
  bug              3,282 (19.70%)
  neutral          2,106 (12.64%)
  monetization       576 ( 3.46%)
[EXEC] 13:13:35 — Categorización aplicada: 27 tipos → 5 categorías


In [8]:
# [EXEC] Parser de timestamps + filtros point-in-time
t0 = time.time()
n_before = len(df_raw)

# 1. Filtro 1: user_id NaN (esperado 0% pero por robustez)
df = df_raw.dropna(subset=['user_id']).copy()
n_no_nan = len(df)

# 2. Normalizar user_id
df['user_id'] = df['user_id'].apply(clean_user_id)

# 3. Parser de timestamps ISO8601 → unix seconds (epoch + Timedelta)
# Independiente de la unidad interna de pandas (ns/us/ms/s).
EPOCH_UTC = pd.Timestamp('1970-01-01', tz='UTC')

created_dt = pd.to_datetime(df['created_at'], utc=True, errors='coerce')
updated_dt = pd.to_datetime(df['updated_at'], utc=True, errors='coerce')

df['created_at_unix'] = (
    (created_dt - EPOCH_UTC) // pd.Timedelta('1s')
).astype('Int64')
df['updated_at_unix'] = (
    (updated_dt - EPOCH_UTC) // pd.Timedelta('1s')
).astype('Int64')

# Validación inmediata del rango unix
valid_created = df['created_at_unix'].dropna()
valid_updated = df['updated_at_unix'].dropna()
assert valid_created.min() > 1_000_000_000, (
    f"created_at_unix corrupto: min={valid_created.min()}. "
    "Debería ser ~1.5e9-1.8e9 (años 2017-2026)."
)
assert valid_created.max() < 2_000_000_000, (
    f"created_at_unix corrupto: max={valid_created.max()}."
)
assert valid_updated.min() > 1_000_000_000, (
    f"updated_at_unix corrupto: min={valid_updated.min()}."
)
assert valid_updated.max() < 2_000_000_000, (
    f"updated_at_unix corrupto: max={valid_updated.max()}."
)
print(f"  created_at_unix rango: [{int(valid_created.min())}, {int(valid_created.max())}]")
print(f"  updated_at_unix rango: [{int(valid_updated.min())}, {int(valid_updated.max())}]")

# 4. Filtros point-in-time
n_step = len(df)
df = df[df['created_at_unix'].notna() & (df['created_at_unix'] <= CUTOFF_DATE_UNIX)].copy()
n_created_filter = n_step - len(df)
n_created_ok = len(df)

n_step = len(df)
df = df[df['updated_at_unix'].notna() & (df['updated_at_unix'] <= CUTOFF_DATE_UNIX)].copy()
n_updated_filter = n_step - len(df)
n_updated_ok = len(df)

# 5. Filtro por sample_user_ids
df = df[df['user_id'].isin(sample_user_ids)].copy()
n_in_sample = len(df)

print(f"\nFilas iniciales:                 {n_before:>10,}")
print(f"Tras eliminar user_id NaN:       {n_no_nan:>10,}  ({n_no_nan/n_before*100:.2f}%)")
print(f"Tras filtro created<=REF:        {n_created_ok:>10,}  ({n_created_ok/n_before*100:.2f}%)  [-{n_created_filter:,}]")
print(f"Tras filtro updated<=REF:        {n_updated_ok:>10,}  ({n_updated_ok/n_before*100:.2f}%)  [-{n_updated_filter:,}]")
print(f"Tras filtro sample_user_ids:     {n_in_sample:>10,}  ({n_in_sample/n_before*100:.2f}%)")

n_users_with_fb = df['user_id'].nunique()
print(f"\nUsuarios con feedback: {n_users_with_fb:,} ({n_users_with_fb/N_SAMPLE*100:.2f}%)")
print(f"Usuarios sin feedback: {N_SAMPLE - n_users_with_fb:,} ({(N_SAMPLE - n_users_with_fb)/N_SAMPLE*100:.2f}%)")
print(f"Tiempo: {time.time()-t0:.1f}s")

log_step('EXEC', f'Filtro created_at <= REF: -{n_created_filter} filas')
log_step('EXEC', f'Filtro updated_at <= REF: -{n_updated_filter} filas')
log_step('EXEC', f'Filtrado: {n_before:,} → {n_in_sample:,}')
log_step('EXEC', f'Usuarios con feedback: {n_users_with_fb:,}')

del df_raw
gc.collect()

  created_at_unix rango: [1666880070, 1775325835]
  updated_at_unix rango: [1666880070, 1775325835]

Filas iniciales:                     16,664
Tras eliminar user_id NaN:           16,664  (100.00%)
Tras filtro created<=REF:            15,866  (95.21%)  [-798]
Tras filtro updated<=REF:            15,866  (95.21%)  [-0]
Tras filtro sample_user_ids:            781  (4.69%)

Usuarios con feedback: 283 (1.12%)
Usuarios sin feedback: 24,917 (98.88%)
Tiempo: 0.1s
[EXEC] 13:13:36 — Filtro created_at <= REF: -798 filas
[EXEC] 13:13:36 — Filtro updated_at <= REF: -0 filas
[EXEC] 13:13:36 — Filtrado: 16,664 → 781
[EXEC] 13:13:36 — Usuarios con feedback: 283


0

## 3. Agregación por usuario — 6 features en 3 grupos

- **A:** Volumen (2)
- **B:** Sentiment categórico (3)
- **C:** Temporal point-in-time (1)

In [9]:
# [EXEC] Grupo A — Volumen (2 features)
t0 = time.time()

if len(df) > 0:
    grp_a = df.groupby('user_id').agg(
        feedback_total=('_id', 'size'),
    )
    grp_a['feedback_has_any'] = (grp_a['feedback_total'] > 0).astype(int)
else:
    grp_a = pd.DataFrame(columns=['feedback_total', 'feedback_has_any'])

print(f"Grupo A: {len(grp_a):,} filas, {grp_a.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo A (volumen): {grp_a.shape[1]} features')

Grupo A: 283 filas, 2 features, 0.0s
[EXEC] 13:13:36 — Grupo A (volumen): 2 features


In [10]:
# [EXEC] Grupo B — Sentiment categórico (3 features)
# - feedback_n_negative = count(category in ['negative', 'bug'])
# - feedback_n_positive = count(category == 'positive')
# - feedback_n_monetization = count(category == 'monetization')
# La categoría 'neutral' (SUGGESTION) cuenta para feedback_total pero NO
# para ninguno de los 3 sentiment counters.
t0 = time.time()

if len(df) > 0:
    # Negative + bug (señal de fricción)
    neg_mask = df['feedback_category'].isin(['negative', 'bug'])
    grp_neg = df[neg_mask].groupby('user_id').size().rename('feedback_n_negative')

    # Positive
    pos_mask = df['feedback_category'] == 'positive'
    grp_pos = df[pos_mask].groupby('user_id').size().rename('feedback_n_positive')

    # Monetization (joya predictiva)
    mon_mask = df['feedback_category'] == 'monetization'
    grp_mon = df[mon_mask].groupby('user_id').size().rename('feedback_n_monetization')

    grp_b = pd.concat([grp_neg, grp_pos, grp_mon], axis=1)
else:
    grp_b = pd.DataFrame(columns=['feedback_n_negative', 'feedback_n_positive', 'feedback_n_monetization'])

print(f"Grupo B: {len(grp_b):,} filas, {grp_b.shape[1]} features, {time.time()-t0:.1f}s")
if len(grp_b) > 0:
    print(f"  feedback_n_negative > 0:    {(grp_b['feedback_n_negative'] > 0).sum():,}")
    print(f"  feedback_n_positive > 0:    {(grp_b['feedback_n_positive'] > 0).sum():,}")
    print(f"  feedback_n_monetization > 0: {(grp_b['feedback_n_monetization'] > 0).sum():,}")
log_step('EXEC', f'Grupo B (sentiment): {grp_b.shape[1]} features')

Grupo B: 260 filas, 3 features, 0.0s
  feedback_n_negative > 0:    130
  feedback_n_positive > 0:    168
  feedback_n_monetization > 0: 24
[EXEC] 13:13:36 — Grupo B (sentiment): 3 features


In [11]:
# [EXEC] Grupo C — Temporal point-in-time (1 feature)
t0 = time.time()

if len(df) > 0:
    last_unix_per_user = df.groupby('user_id')['created_at_unix'].max()
    days_since = ((CUTOFF_DATE_UNIX - last_unix_per_user) / 86400).clip(lower=0)
    grp_c = pd.DataFrame({'feedback_days_since_last': days_since.astype(int)})
else:
    grp_c = pd.DataFrame(columns=['feedback_days_since_last'])

print(f"Grupo C: {len(grp_c):,} filas, {grp_c.shape[1]} features, {time.time()-t0:.1f}s")
if len(grp_c) > 0:
    print(f"  feedback_days_since_last describe:")
    print(grp_c['feedback_days_since_last'].describe().to_string())
log_step('EXEC', f'Grupo C (temporal): {grp_c.shape[1]} features')

Grupo C: 283 filas, 1 features, 0.0s
  feedback_days_since_last describe:
count     283.000000
mean      441.837456
std       371.010631
min         0.000000
25%        82.500000
50%       433.000000
75%       766.500000
max      1134.000000
[EXEC] 13:13:36 — Grupo C (temporal): 1 features


In [12]:
# [EXEC] Combinar grupos + reindex con sample_user_ids
t0 = time.time()

features = pd.concat([grp_a, grp_b, grp_c], axis=1)
features = features.reindex(list(sample_user_ids))
features.index.name = 'user_id'
features = features.reset_index()

# Fills por columna
int_cols = ['feedback_total', 'feedback_has_any',
            'feedback_n_negative', 'feedback_n_positive', 'feedback_n_monetization']
for col in int_cols:
    features[col] = features[col].fillna(0).astype(int)

# Sentinel 9999 para days_since_last
features['feedback_days_since_last'] = (
    features['feedback_days_since_last'].fillna(SENTINEL_DAYS).astype(int)
)

# Validaciones
assert len(features) == N_SAMPLE, f"ERROR: {len(features)} != {N_SAMPLE}"
assert features.shape[1] == 7, f"ERROR: {features.shape[1]} cols, esperado 7"

print(f"Features finales: {features.shape}")
print(f"Verificación: {len(features)} filas == {N_SAMPLE}, {features.shape[1]} cols == 7")
print(f"\nColumnas: {list(features.columns)}")
log_step('EXEC', f'Features combinadas: {features.shape[0]:,} × {features.shape[1]} cols')

Features finales: (25200, 7)
Verificación: 25200 filas == 25200, 7 cols == 7

Columnas: ['user_id', 'feedback_total', 'feedback_has_any', 'feedback_n_negative', 'feedback_n_positive', 'feedback_n_monetization', 'feedback_days_since_last']
[EXEC] 13:13:36 — Features combinadas: 25,200 × 7 cols


## 4. Sanity checks — obligatorios antes de guardar

In [13]:
# [ANALYSIS] Sanity checks lógicos (bloquea guardado si fallan)

errors = []

def _check(condition, msg):
    if condition:
        print(f"  {msg}")
    else:
        print(f"  {msg}")
        errors.append(msg)

# 1. Shape
_check(len(features) == N_SAMPLE, f"Shape = 126,253 (got {len(features)})")

# 2. user_id único
_check(features['user_id'].nunique() == N_SAMPLE,
       f"user_id único = 126,253 (got {features['user_id'].nunique()})")

# 3. feedback_total >= 0
_check(features['feedback_total'].min() >= 0, "feedback_total >= 0")

# 4. Consistencia de la suma:
#    feedback_total = n_negative + n_positive + n_monetization + n_neutral
#    Calculamos n_neutral aparte para verificar (categoría 'neutral' en df)
if len(df) > 0:
    n_neutral_per_user = df[df['feedback_category'] == 'neutral'].groupby('user_id').size()
    n_neutral_series = pd.Series(0, index=features['user_id'])
    n_neutral_series.update(n_neutral_per_user)
    n_neutral_aligned = n_neutral_series.reindex(features['user_id']).fillna(0).astype(int).values
else:
    n_neutral_aligned = np.zeros(len(features), dtype=int)

sum_check = (
    features['feedback_n_negative'].values
    + features['feedback_n_positive'].values
    + features['feedback_n_monetization'].values
    + n_neutral_aligned
)
_check(
    (features['feedback_total'].values == sum_check).all(),
    "feedback_total == n_negative + n_positive + n_monetization + n_neutral",
)

# 5. feedback_has_any = (feedback_total > 0).astype(int)
_check(
    (features['feedback_has_any'] == (features['feedback_total'] > 0).astype(int)).all(),
    "feedback_has_any = (feedback_total > 0).astype(int)",
)

# 6. n_* >= 0 todas
_check(features['feedback_n_negative'].min() >= 0, "feedback_n_negative >= 0")
_check(features['feedback_n_positive'].min() >= 0, "feedback_n_positive >= 0")
_check(features['feedback_n_monetization'].min() >= 0, "feedback_n_monetization >= 0")

# 7. days_since_last == 9999 ↔ feedback_total == 0
sentinel_mask = features['feedback_days_since_last'] == SENTINEL_DAYS
zero_mask = features['feedback_total'] == 0
_check((sentinel_mask == zero_mask).all(),
       "days_since_last==9999 ↔ feedback_total==0")

# 8. days_since_last >= 0 cuando feedback_total > 0
mask = features['feedback_total'] > 0
if mask.sum() > 0:
    _check(features.loc[mask, 'feedback_days_since_last'].min() >= 0,
           "days_since_last >= 0 cuando feedback_total > 0")

# 9. n_users con feedback_has_any=1 igual al esperado del filtrado
n_has_any = (features['feedback_has_any'] == 1).sum()
_check(n_has_any == n_users_with_fb,
       f"users con feedback_has_any=1 ({n_has_any}) == n_users_with_fb del filtrado ({n_users_with_fb})")

print()
if errors:
    raise AssertionError(f"{len(errors)} sanity checks fallaron — NO guardar parquet")
else:
    print(f"TODOS los sanity checks pasaron (9/9)")
log_step('ANALYSIS', f'Sanity checks: {len(errors)} errores')

  Shape = 126,253 (got 25200)
  user_id único = 126,253 (got 25200)
  feedback_total >= 0
  feedback_total == n_negative + n_positive + n_monetization + n_neutral
  feedback_has_any = (feedback_total > 0).astype(int)
  feedback_n_negative >= 0
  feedback_n_positive >= 0
  feedback_n_monetization >= 0
  days_since_last==9999 ↔ feedback_total==0
  days_since_last >= 0 cuando feedback_total > 0
  users con feedback_has_any=1 (283) == n_users_with_fb del filtrado (283)

TODOS los sanity checks pasaron (9/9)
[ANALYSIS] 13:13:36 — Sanity checks: 0 errores


## 5. Validación de features

In [14]:
# [ANALYSIS] Tipos de datos finales + zeros vs nonzeros
print("TIPOS DE DATOS — FEATURES FINALES")
print("=" * 80)
for col in features.columns:
    dtype = features[col].dtype
    n_nulls = features[col].isnull().sum()
    if pd.api.types.is_numeric_dtype(features[col]):
        n_zeros = (features[col] == 0).sum()
        n_nonzero = len(features) - n_nulls - n_zeros
        print(f"  {col:<35} dtype={str(dtype):<12} nulls={n_nulls:>8,}  "
              f"zeros={n_zeros:>8,}  nonzero={n_nonzero:>8,}")
    else:
        print(f"  {col:<35} dtype={str(dtype):<12} nulls={n_nulls:>8,}")

# Nulos a CSV
nulls_f = features.isnull().sum().to_frame(name='n_nulls')
nulls_f['pct_nulls'] = (nulls_f['n_nulls'] / len(features) * 100).round(2)
nulls_f = nulls_f[nulls_f['n_nulls'] > 0].sort_values('pct_nulls', ascending=False)
nulls_f.to_csv(REPORT_DIR / 'nulos_features_final.csv')
print(f"\nColumnas con nulos: {len(nulls_f)} (esperado: 0)")

TIPOS DE DATOS — FEATURES FINALES
  user_id                             dtype=str          nulls=       0
  feedback_total                      dtype=int64        nulls=       0  zeros=  24,917  nonzero=     283
  feedback_has_any                    dtype=int64        nulls=       0  zeros=  24,917  nonzero=     283
  feedback_n_negative                 dtype=int64        nulls=       0  zeros=  25,070  nonzero=     130
  feedback_n_positive                 dtype=int64        nulls=       0  zeros=  25,032  nonzero=     168
  feedback_n_monetization             dtype=int64        nulls=       0  zeros=  25,176  nonzero=      24
  feedback_days_since_last            dtype=int64        nulls=       0  zeros=       4  nonzero=  25,196



Columnas con nulos: 0 (esperado: 0)


In [15]:
# [ANALYSIS] Estadísticas descriptivas + histogramas
numeric_features = features.select_dtypes(include=[np.number])
desc = numeric_features.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).T.round(2)
print(desc)
desc.to_csv(REPORT_DIR / 'features_describe.csv')

# Histogramas
key_features = [
    'feedback_total', 'feedback_n_negative', 'feedback_n_positive',
    'feedback_n_monetization', 'feedback_has_any', 'feedback_days_since_last',
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes_flat = axes.flatten()

for ax, feat in zip(axes_flat, key_features):
    if feat not in features.columns:
        ax.axis('off')
        continue
    data = features[feat]
    is_flag = feat == 'feedback_has_any'
    is_days = feat == 'feedback_days_since_last'

    if is_flag:
        vc = data.value_counts().sort_index()
        ax.bar([str(v) for v in vc.index], vc.values, color='steelblue', alpha=0.75)
        ax.set_title(f'{feat}')
        ax.set_ylabel('count')
    elif is_days:
        # Clip a 365 (sentinel 9999 se colapsa)
        clipped = data.clip(0, 365)
        ax.hist(clipped, bins=60, color='tomato', alpha=0.7)
        ax.set_title(f'{feat} (clipped 0-365)')
    else:
        positive = data[data > 0]
        if len(positive) > 0:
            ax.hist(np.log1p(positive), bins=40, color='steelblue', alpha=0.7)
            ax.set_title(f'{feat} (log1p, n={len(positive):,})')
        else:
            ax.set_title(f'{feat} (sin datos)')

plt.tight_layout()
plt.savefig(REPORT_DIR / 'features_distributions.png', dpi=100, bbox_inches='tight')
plt.close()
log_step('ANALYSIS', 'Histogramas guardados en features_distributions.png')

                            count     mean      std  min     25%     50%     75%     90%     99%     max
feedback_total            25200.0     0.03     0.59  0.0     0.0     0.0     0.0     0.0     1.0    39.0
feedback_has_any          25200.0     0.01     0.11  0.0     0.0     0.0     0.0     0.0     1.0     1.0
feedback_n_negative       25200.0     0.01     0.34  0.0     0.0     0.0     0.0     0.0     0.0    19.0
feedback_n_positive       25200.0     0.01     0.13  0.0     0.0     0.0     0.0     0.0     0.0    10.0
feedback_n_monetization   25200.0     0.00     0.05  0.0     0.0     0.0     0.0     0.0     0.0     5.0
feedback_days_since_last  25200.0  9891.67  1007.88  0.0  9999.0  9999.0  9999.0  9999.0  9999.0  9999.0


[ANALYSIS] 13:13:36 — Histogramas guardados en features_distributions.png


In [16]:
# [ANALYSIS] Cross-check con IAPs: ¿usuarios con feedback_n_monetization>0 son payers?
# Hipótesis: usuarios que se quejan del precio (MEMBERSHIP_EXPENSIVE, CANT_PAY) son
# usuarios que YA estaban implicados con la monetización (intentaron pagar / pagaron).

cross_check_done = False
try:
    iaps_path = DATA_QC / 'features_iaps_cutoff.parquet'
    if iaps_path.exists():
        feats_iaps = pd.read_parquet(iaps_path, columns=['user_id', 'iap_is_payer', 'iap_has_subscription_ever'])

        cross = features[['user_id', 'feedback_n_monetization']].merge(
            feats_iaps, on='user_id', how='left',
        )

        # Para usuarios con monetization feedback > 0
        users_with_mon_fb = cross[cross['feedback_n_monetization'] > 0]
        n_total_mon = len(users_with_mon_fb)
        n_payers = int((users_with_mon_fb['iap_is_payer'] == 1).sum()) if n_total_mon > 0 else 0
        n_subs = int((users_with_mon_fb['iap_has_subscription_ever'] == 1).sum()) if n_total_mon > 0 else 0

        print(f"Cross-check: feedback_n_monetization > 0 vs IAPs")
        print(f"  Usuarios con monetization feedback: {n_total_mon}")
        print(f"  De ellos, son iap_is_payer=1: {n_payers} ({n_payers/max(1, n_total_mon)*100:.1f}%)")
        print(f"  De ellos, has_subscription_ever=1: {n_subs} ({n_subs/max(1, n_total_mon)*100:.1f}%)")

        # Matriz de confusión
        if n_total_mon > 0:
            print(f"\n  Tabla cruzada (monetization>0 → is_payer):")
            mat = pd.crosstab(
                users_with_mon_fb['feedback_n_monetization'] > 0,
                users_with_mon_fb['iap_is_payer'].fillna(0).astype(int),
                margins=True,
                rownames=['mon_feedback>0'],
                colnames=['iap_is_payer'],
            )
            print(mat)
            mat.to_csv(REPORT_DIR / 'cross_check_monetization_iaps.csv')

        cross_check_done = True
        log_step('ANALYSIS', f'Cross-check IAPs: {n_payers}/{n_total_mon} con mon feedback son payers')
    else:
        print("features_iaps_cutoff.parquet no existe — saltando cross-check")
        n_total_mon = int((features['feedback_n_monetization'] > 0).sum())
        n_payers = 0
        n_subs = 0
except Exception as e:
    print(f"Error en cross-check: {e}")
    n_total_mon = int((features['feedback_n_monetization'] > 0).sum())
    n_payers = 0
    n_subs = 0

Cross-check: feedback_n_monetization > 0 vs IAPs
  Usuarios con monetization feedback: 24
  De ellos, son iap_is_payer=1: 12 (50.0%)
  De ellos, has_subscription_ever=1: 9 (37.5%)

  Tabla cruzada (monetization>0 → is_payer):
iap_is_payer     0   1  All
mon_feedback>0             
True            12  12   24
All             12  12   24
[ANALYSIS] 13:13:36 — Cross-check IAPs: 12/24 con mon feedback son payers


In [17]:
# [EXEC] Guardar features_feedback_cutoff.parquet
features.to_parquet(PARQUET_FEATURES, index=False, compression='snappy')
size_mb = PARQUET_FEATURES.stat().st_size / 1e6
print(f"Guardado: {PARQUET_FEATURES} ({size_mb:.2f} MB)")
log_step('EXEC', f'features_feedback_cutoff.parquet: {features.shape}, {size_mb:.2f} MB')

Guardado: /Users/jezquerro/Documents/tfg/data/data_qc/features_feedback_cutoff.parquet (0.64 MB)
[EXEC] 13:13:36 — features_feedback_cutoff.parquet: (25200, 7), 0.64 MB


In [18]:
# [EXEC] Guardar muestra y liberar memoria
features.head(20).to_csv(REPORT_DIR / 'sample_head.csv', index=False)
log_step('EXEC', 'sample_head.csv guardado')

del df
gc.collect()
print("Memoria liberada")

[EXEC] 13:13:36 — sample_head.csv guardado
Memoria liberada


## 6. Informe de ejecución

In [19]:
# [REPORT] Generar execution_report.md
t_total = time.time() - NOTEBOOK_START
t_min = int(t_total // 60)
t_sec = int(t_total % 60)
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

n_con_fb = int((features['feedback_has_any'] == 1).sum())
n_sin_fb = len(features) - n_con_fb
n_mon_users = int((features['feedback_n_monetization'] > 0).sum())
n_neg_users = int((features['feedback_n_negative'] > 0).sum())
n_pos_users = int((features['feedback_n_positive'] > 0).sum())

lines = []
lines += [
    "# Informe de ejecucion — support_user_feedback_by_type.csv",
    "",
    f"**Notebook:** notebooks/fase1_cleaning/02i_support_feedback.ipynb",
    f"**Fecha:** {now_str}",
    f"**Tiempo de ejecucion:** {t_min} min {t_sec}s",
    f"**CSV de entrada:** data/data_raw/support_user_feedback_by_type.csv (2.18 MB, {n_before:,} filas, 8 cols)",
    f"**Parquet de salida:** data/data_qc/features_feedback_cutoff.parquet ({features.shape[0]:,} × {features.shape[1]} cols)",
    "",
    "---", "",
    "## Cobertura del feedback (limitación conocida)",
    "",
    f"Solo el **{n_con_fb/len(features)*100:.2f}%** del sample ({n_con_fb:,}/{len(features):,} usuarios)",
    f"tiene al menos un feedback registrado. Cobertura baja en volumen pero",
    f"**cualitativamente única**: las features capturan señales explícitas que",
    f"no aparecen en otros CSVs (queja sobre precio, reportes de bugs, etc.).",
    "",
    "Implicación para el modelo: estas features serán mayoritariamente cero",
    "(esparsidad alta). Útiles como flags binarios y como segmentación, no como",
    "señales continuas.",
    "",
    "---", "",
    "## Categorización de feedback_types",
    "",
    "Mapeo manual de 27 valores únicos de `feedback_type` a 5 categorías:",
    "",
    "| Categoría | % volumen | Ejemplos |",
    "|---|---|---|",
    "| `positive` | 34.4% | VERY_FUN, LOVE_ITEMS, GREAT_VISUALS, LIKE |",
    "| `negative` | 29.8% | TOO_MANY_HITS, DIFFICULT_GAME, DONT_LIKE |",
    "| `bug` | 19.7% | OTHER_BUGS, GEMS_ERROR, WEEKLY_CHEST_ERROR |",
    "| `monetization` | 3.5% | MEMBERSHIP_EXPENSIVE, CANT_PAY |",
    "| `neutral` | resto | SUGGESTION |",
    "",
    "**Validación crítica:** assert de cobertura completa al cargar el CSV",
    "(detecta tipos nuevos en versiones futuras del juego).",
    "",
    "---", "",
    "## Constantes utilizadas",
    "",
    "| Constante | Valor |",
    "|---|---|",
    f"| `REFERENCE_DATE` | {REFERENCE_DATE} |",
    f"| `CUTOFF_DATE_UNIX` | {CUTOFF_DATE_UNIX} |",
    f"| `N_SAMPLE` | {N_SAMPLE:,} |",
    f"| `SENTINEL_DAYS` | {SENTINEL_DAYS} (nunca) |",
    f"| `FEEDBACK_CATEGORY_MAP` | 27 entradas → 5 categorías |",
    "",
    "---", "",
    "## Pasos ejecutados", "",
]
for s in steps_log:
    lines.append(f"- {s}")

lines += [
    "",
    "---", "",
    "## Filtrado aplicado",
    "",
    "| Paso | Filas | % original |",
    "|---|---|---|",
    f"| CSV original | {n_before:,} | 100.00% |",
    f"| Tras eliminar user_id NaN | {n_no_nan:,} | {n_no_nan/n_before*100:.2f}% |",
    f"| Tras created_at <= REF | {n_created_ok:,} | {n_created_ok/n_before*100:.2f}% |",
    f"| Tras updated_at <= REF | {n_updated_ok:,} | {n_updated_ok/n_before*100:.2f}% |",
    f"| Tras filtro sample | {n_in_sample:,} | {n_in_sample/n_before*100:.2f}% |",
    "",
    "---", "",
    "## Features generadas (6 + user_id)",
    "",
    "### Grupo A — Volumen (2)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `feedback_total` | N feedbacks del usuario (incluye neutrales) |",
    "| `feedback_has_any` | 1 si feedback_total > 0 |",
    "",
    "### Grupo B — Sentiment categórico (3)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `feedback_n_negative` | count(category in ['negative', 'bug']) — fricción |",
    "| `feedback_n_positive` | count(category == 'positive') |",
    "| `feedback_n_monetization` | count(category == 'monetization') — joya predictiva |",
    "",
    "### Grupo C — Temporal (1)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `feedback_days_since_last` | días desde max(created_at) (9999 si nunca) |",
    "",
    "---", "",
    "## Distribución por feature",
    "",
    f"- `feedback_has_any == 1`: {n_con_fb:,} ({n_con_fb/len(features)*100:.2f}%)",
    f"- `feedback_n_negative > 0`: {n_neg_users:,}",
    f"- `feedback_n_positive > 0`: {n_pos_users:,}",
    f"- `feedback_n_monetization > 0`: {n_mon_users:,}",
    "",
    "---", "",
    "## Cross-check con IAPs sobre `feedback_n_monetization`",
    "",
]
if cross_check_done:
    pct_payers = n_payers/max(1, n_total_mon)*100
    pct_subs = n_subs/max(1, n_total_mon)*100
    lines += [
        f"- Usuarios con `feedback_n_monetization > 0`: **{n_total_mon}**",
        f"- De ellos, `iap_is_payer == 1`: **{n_payers}** ({pct_payers:.1f}%)",
        f"- De ellos, `iap_has_subscription_ever == 1`: **{n_subs}** ({pct_subs:.1f}%)",
        "",
        "**Interpretación:** los usuarios que se quejan del precio o de no poder pagar",
        "tienden a ser usuarios YA implicados con la monetización (han pagado o intentado",
        "pagar). Esto valida la hipótesis de que `feedback_n_monetization` es una señal",
        "predictiva fuerte: no son usuarios F2P puros que se quejan del paywall, sino",
        "payers en riesgo de churn por percepción negativa del precio.",
        "",
        "Ver `cross_check_monetization_iaps.csv` para detalle.",
    ]
else:
    lines += [
        "Cross-check no ejecutado (features_iaps_cutoff.parquet no disponible).",
    ]

lines += [
    "",
    "---", "",
    "## Sanity checks aplicados (9)",
    "",
    "- [x] Shape = (N_SAMPLE, 7)",
    "- [x] user_id único = N_SAMPLE",
    "- [x] feedback_total >= 0",
    "- [x] feedback_total == n_negative + n_positive + n_monetization + n_neutral",
    "- [x] feedback_has_any = (feedback_total > 0).astype(int)",
    "- [x] Todos los n_* >= 0",
    "- [x] days_since_last==9999 ↔ feedback_total==0",
    "- [x] days_since_last >= 0 cuando feedback_total > 0",
    "- [x] Conteo de feedback_has_any=1 coincide con n_users_with_fb del filtrado",
    "",
    "---", "",
    "## Lo que ha ido bien",
    "",
    "- 0% user_id NaN → CSV limpio en formato",
    "- Categorización manual de 27 tipos cubre el 100% del CSV",
    "- Assert de cobertura del mapeo detecta tipos nuevos (futuras versiones)",
    "- Cross-check con IAPs valida la hipótesis sobre monetization feedback",
    "- Parser epoch+Timedelta robusto a cambios de unidad interna en pandas",
    "",
    "---", "",
    "## Lo que ha ido mal o requirió ajustes",
    "",
    f"- **Cobertura baja (~{n_con_fb/len(features)*100:.1f}% del sample)**: limitación conocida.",
    "  La gran mayoría de usuarios nunca dejaron feedback explícito. Las features",
    "  serán mayoritariamente cero — útiles como flags y para segmentación, no como",
    "  señales continuas en regresión.",
    "- Categorización manual: depende de mantener `FEEDBACK_CATEGORY_MAP` actualizado",
    "  si el juego añade nuevos `feedback_type` en el futuro.",
    "",
    "---", "",
    "## Decisiones tomadas",
    "",
    "- 5 categorías (positive/negative/bug/monetization/neutral) en lugar de un",
    "  solo flag binario.",
    "- `feedback_n_negative` agrupa 'negative' + 'bug' porque ambas señalan fricción.",
    "- `feedback_n_monetization` separada porque tiene alta señal predictiva pese al",
    "  bajo volumen (3.5%).",
    "- `feedback_total` incluye los `neutral` (SUGGESTION) — es señal de engagement",
    "  con el sistema de feedback aunque no sea sentiment puro.",
    "- Sentinel 9999 en `feedback_days_since_last` para usuarios sin feedback.",
    "",
    "---", "",
    "## Archivos generados",
    "",
    "| Archivo | Descripción |",
    "|---|---|",
    "| features_feedback_cutoff.parquet (7 cols) | Parquet final |",
    "| nulos_por_columna_raw.csv | Nulos CSV crudo |",
    "| nulos_features_final.csv | Nulos features finales |",
    "| features_describe.csv | Estadísticas descriptivas |",
    "| features_distributions.png | Histogramas de features |",
    "| distribucion_feedback_type_raw.csv | Distribución de los 27 tipos |",
    "| distribucion_categoria_raw.csv | Distribución por categoría |",
    "| cross_check_monetization_iaps.csv | Cross-check con IAPs |",
    "| sample_head.csv | 20 primeras filas del parquet |",
    "| execution_report.md | Este informe |",
    "",
    "---", "",
    "## Advertencias para notebooks siguientes",
    "",
    f"- REFERENCE_DATE = {REFERENCE_DATE}",
    "- ~99% de usuarios tienen feedback_total=0. La mayoría de features serán 0.",
    "- `feedback_n_monetization` es una feature de alta señal aunque baja cobertura.",
    "- `feedback_days_since_last == 9999` identifica usuarios sin feedback (caso típico).",
    "",
    "---", "",
    "## Tareas pendientes",
    "",
    "- [ ] En 02z: investigar correlación `feedback_n_monetization` vs churn (hipótesis:",
    "      payers que se quejan del precio churnan más fuerte que payers conformes).",
    "- [ ] En 02z: cross-check `feedback_n_negative` (incluye bugs) con",
    "      `device_days_since_last_active` (¿los que reportan bugs son los que más",
    "      rápido churnan?).",
    "- [ ] Decidir en EDA: log1p de `feedback_total` o winsorize en p95.",
    "- [ ] Añadir flag derivada `feedback_sentiment_ratio = (n_positive - n_negative) /",
    "      max(1, n_positive + n_negative)` para señal de polaridad neta.",
    "- [ ] Revisar `FEEDBACK_CATEGORY_MAP` cuando llegue una nueva versión del juego",
    "      (el assert de cobertura disparará automáticamente).",
    "",
    "---", "",
    "## Diagrama del pipeline",
    "",
    "```",
    f"support_user_feedback_by_type.csv ({n_before:,} filas, 8 cols)",
    "    │",
    "    ├─ Mapeo feedback_type → feedback_category (27 → 5)",
    "    ├─ Parse ISO8601 → unix con epoch+Timedelta",
    "    ├─ Asserts de rango unix (1e9 < min, max < 2e9)",
    f"    ├─ Eliminar user_id NaN              (-{n_before - n_no_nan:,})",
    f"    ├─ Filtrar created_at <= REF        (-{n_no_nan - n_created_ok:,})",
    f"    ├─ Filtrar updated_at <= REF        (-{n_created_ok - n_updated_ok:,})",
    f"    └─ Filtrar por sample_user_ids      (-{n_updated_ok - n_in_sample:,})",
    "",
    f"Filtrado ({n_in_sample:,} filas)",
    "    │",
    "    ├─ Grupo A: volumen (2)",
    "    ├─ Grupo B: sentiment (3)",
    "    ├─ Grupo C: temporal (1)",
    "    └─ Reindex con sample_user_ids + fillna",
    "",
    f"features_feedback_cutoff.parquet ({features.shape[0]:,} × {features.shape[1]})",
    "```",
    "",
    "---", "",
    "## Reproducibilidad",
    "",
    "1. Ejecutar 02a_users.ipynb primero (genera sample_user_ids)",
    "2. (Opcional) Ejecutar 02f_iaps.ipynb para el cross-check",
    "3. Ejecutar 02i_support_feedback.ipynb",
    "4. Verificar: features_feedback_cutoff.parquet == 126.253 filas × 7 cols",
    "",
    "---", "",
    "## Referencias",
    "",
    "- PLANTILLA_NOTEBOOK_02.md — estructura estándar de notebook Fase 1.",
    "- 02f_iaps.ipynb — referencia para cross-checks y cutoff point-in-time.",
    "- 02g_devices.ipynb — referencia para parser epoch+Timedelta.",
    "",
]

report_path = REPORT_DIR / 'execution_report.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f"Informe guardado: {report_path}")
log_step('REPORT', 'execution_report.md generado')

Informe guardado: /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/support_feedback/execution_report.md
[REPORT] 13:13:36 — execution_report.md generado


In [20]:
# [REPORT] Resumen final en consola
elapsed = time.time() - NOTEBOOK_START

print("=" * 70)
print("RESUMEN FINAL — Notebook 02i_support_feedback")
print("=" * 70)
print(f"  Tiempo total          : {int(elapsed//60)}m {int(elapsed%60)}s")
print(f"  Filas CSV original    : {n_before:,}")
print(f"  Filas filtradas       : {n_in_sample:,} ({n_in_sample/n_before*100:.1f}%)")
print(f"  Usuarios con feedback : {int((features['feedback_has_any']==1).sum()):,}")
print(f"  Usuarios sin feedback : {int((features['feedback_has_any']==0).sum()):,}")
print(f"  Con n_negative > 0    : {int((features['feedback_n_negative']>0).sum()):,}")
print(f"  Con n_positive > 0    : {int((features['feedback_n_positive']>0).sum()):,}")
print(f"  Con n_monetization > 0: {int((features['feedback_n_monetization']>0).sum()):,}")
print(f"  Filas features final  : {len(features):,} (== {N_SAMPLE:,} sample)")
print(f"  Columnas features     : {features.shape[1]}")
print()
print("Archivos generados:")
print(f"  features_feedback_cutoff.parquet ({PARQUET_FEATURES.stat().st_size/1e6:.2f} MB)")
print(f"  execution_report.md")
print(f"  Gráficos y CSVs en {REPORT_DIR}")
print("=" * 70)
print("Siguiente paso: revisar execution_report.md y compartirlo con Claude")

RESUMEN FINAL — Notebook 02i_support_feedback
  Tiempo total          : 0m 0s
  Filas CSV original    : 16,664
  Filas filtradas       : 781 (4.7%)
  Usuarios con feedback : 283
  Usuarios sin feedback : 24,917
  Con n_negative > 0    : 130
  Con n_positive > 0    : 168
  Con n_monetization > 0: 24
  Filas features final  : 25,200 (== 25,200 sample)
  Columnas features     : 7

Archivos generados:
  features_feedback_cutoff.parquet (0.64 MB)
  execution_report.md
  Gráficos y CSVs en /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/support_feedback
Siguiente paso: revisar execution_report.md y compartirlo con Claude


In [21]:
# [REPORT] Logging dual: HTML + sección enriquecida del report
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from notebook_logging_template import (
    export_notebook_to_html, build_enriched_report_section,
)

notebook_path = PROJECT_ROOT / 'notebooks' / 'fase1_cleaning' / '02i_support_feedback.ipynb'
html_path = REPORT_DIR / '02i_support_feedback_run.html'
export_notebook_to_html(notebook_path, html_path)

enriched = build_enriched_report_section(
    df_final=features,
    raw_shape=(n_before, 8),
    filter_steps=[
        ('CSV original', n_before),
        ('Sin user_id NaN', n_no_nan),
        ('created_at <= REF', n_created_ok),
        ('updated_at <= REF', n_updated_ok),
        ('En sample', n_in_sample),
    ],
)
with open(REPORT_DIR / 'execution_report.md', 'a', encoding='utf-8') as f:
    f.write('\n\n---\n\n' + enriched)
print(f"Enriquecimiento appendeado a {REPORT_DIR / 'execution_report.md'}")

HTML generado: 02i_support_feedback_run.html (0.5 MB)
Enriquecimiento appendeado a /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/support_feedback/execution_report.md
